Cell 1: Imports

In [ ]:
# Cell 1: Imports

import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

Cell 2: Config / Paths (full split version)

In [ ]:
# Cell 2: Config / Paths (full split version)


import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ROOT = Path.cwd().parents[1]
print("ROOT:", ROOT)

# ===== full cleaned data =====
CLEAN_DIR = ROOT / "data" / "processed" / "clean_chunks22"

# ===== canonical split =====
PREDS_DIR = ROOT / "results" / "preds"
SPLIT_TRAIN_SEQIDS_PATH = PREDS_DIR / "split_train_seq_ids.npy"
SPLIT_TEST_SEQIDS_PATH = PREDS_DIR / "split_test_seq_ids.npy"

# ===== class names =====
TOF_DIR = ROOT / "data" / "processed_data_tof"
CLASSES_PATH = TOF_DIR / "gesture_classes_raw.npy"

print("CLEAN_DIR:", CLEAN_DIR)
print("SPLIT_TRAIN_SEQIDS_PATH:", SPLIT_TRAIN_SEQIDS_PATH)
print("SPLIT_TEST_SEQIDS_PATH:", SPLIT_TEST_SEQIDS_PATH)
print("CLASSES_PATH:", CLASSES_PATH)

Cell 3: Load clean full data + canonical split

In [ ]:
# Cell 3: Load clean full data + canonical split

clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
print("clean files:", clean_files)

dfs = []
for fp in clean_files:
    df_part = pd.read_parquet(fp)
    print(fp.name, df_part.shape)
    dfs.append(df_part)

full_df = pd.concat(dfs, axis=0, ignore_index=True)

train_seqids = np.load(SPLIT_TRAIN_SEQIDS_PATH, allow_pickle=True)
test_seqids = np.load(SPLIT_TEST_SEQIDS_PATH, allow_pickle=True)

classes = np.load(CLASSES_PATH, allow_pickle=True)

print("full_df shape:", full_df.shape)
print("len(train_seqids):", len(train_seqids))
print("len(test_seqids):", len(test_seqids))
print("classes:", classes)

Cell 4: Inspect columns

In [ ]:
# Cell 4: Inspect columns

print("full_df columns:")
print(full_df.columns.tolist())

tof_cols = [c for c in full_df.columns if str(c).startswith("tof_")]
print("\nNumber of TOF columns:", len(tof_cols))
print("First 10 TOF cols:", tof_cols[:10])

candidate_seq_cols = ["sequence_id", "seq_id", "sequence_counter"]
candidate_label_cols = ["gesture", "label", "target", "behavior"]

seq_col = None
label_col = None

for c in candidate_seq_cols:
    if c in full_df.columns:
        seq_col = c
        break

for c in candidate_label_cols:
    if c in full_df.columns:
        label_col = c
        break

print("\nDetected seq_col:", seq_col)
print("Detected label_col:", label_col)

assert seq_col is not None, "No sequence ID column found."
assert label_col is not None, "No label column found."
assert len(tof_cols) == 320, f"Expected 320 TOF cols, got {len(tof_cols)}"

Cell 5: Build full-split sequences + labels

In [ ]:
# Cell 5: Build full-split sequences + labels

def build_sequence_list_and_labels(df, seq_ids, seq_col, feature_cols, label_col):
    seq_map = {}
    label_map = {}

    grouped = df.groupby(seq_col, sort=False)

    for sid, g in grouped:
        sid = str(sid)
        seq_map[sid] = g[feature_cols].to_numpy(dtype=np.float32)

        labels = g[label_col].astype(str).unique()
        assert len(labels) == 1, f"{sid} has multiple labels: {labels}"
        label_map[sid] = labels[0]

    sequences = []
    labels = []
    kept_seqids = []
    missing = []

    for sid in seq_ids:
        sid = str(sid)
        if sid in seq_map and sid in label_map:
            sequences.append(seq_map[sid])
            labels.append(label_map[sid])
            kept_seqids.append(sid)
        else:
            missing.append(sid)

    return sequences, labels, kept_seqids, missing


X_train_seq, y_train_str, seqid_train_used, missing_train = build_sequence_list_and_labels(
    full_df, train_seqids, seq_col, tof_cols, label_col
)

X_test_seq, y_test_str, seqid_test_used, missing_test = build_sequence_list_and_labels(
    full_df, test_seqids, seq_col, tof_cols, label_col
)

print("Train sequences built:", len(X_train_seq))
print("Test sequences built:", len(X_test_seq))
print("Missing train:", len(missing_train))
print("Missing test:", len(missing_test))

if len(X_train_seq) > 0:
    print("First train sequence shape:", X_train_seq[0].shape)
    print("First train seqid:", seqid_train_used[0])
    print("First train label:", y_train_str[0])

print("First 10 missing train:", missing_train[:10])
print("First 10 missing test:", missing_test[:10])

Cell 6: Convert labels to ids

In [ ]:
# Cell 6: Convert labels to ids

label2id = {str(c): i for i, c in enumerate(classes)}
id2label = {i: str(c) for i, c in enumerate(classes)}

y_train = np.array([label2id[str(y)] for y in y_train_str], dtype=np.int64)
y_test = np.array([label2id[str(y)] for y in y_test_str], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("First 10 y_train:", y_train[:10])

Cell 7: Inspect sequence lengths

In [ ]:
# Cell 7: Inspect sequence lengths

train_lengths = np.array([seq.shape[0] for seq in X_train_seq], dtype=np.int32)
test_lengths = np.array([seq.shape[0] for seq in X_test_seq], dtype=np.int32)

print("Train length stats:")
print("min:", train_lengths.min())
print("max:", train_lengths.max())
print("mean:", train_lengths.mean())
print("median:", np.median(train_lengths))

print("\nTest length stats:")
print("min:", test_lengths.min())
print("max:", test_lengths.max())
print("mean:", test_lengths.mean())
print("median:", np.median(test_lengths))

Cell 8: Reshape TOF sequences into frame tokens for CNN encoder

Each frame is reshaped from 320 to (5, 8, 8).

In [ ]:
# Cell 8: Reshape TOF sequences into frame tokens for CNN encoder

def reshape_tof_sequence(seq_2d: np.ndarray) -> np.ndarray:
    """
    seq_2d: (T, 320)
    return: (T, 5, 8, 8)
    """
    assert seq_2d.ndim == 2 and seq_2d.shape[1] == 320, f"Expected (T, 320), got {seq_2d.shape}"
    return seq_2d.reshape(seq_2d.shape[0], 5, 8, 8).astype(np.float32)

X_train_frames = [reshape_tof_sequence(seq) for seq in X_train_seq]
X_test_frames = [reshape_tof_sequence(seq) for seq in X_test_seq]

print("Train frame sequences:", len(X_train_frames))
print("Test frame sequences:", len(X_test_frames))
print("First train frame sequence shape:", X_train_frames[0].shape)
print("First test frame sequence shape:", X_test_frames[0].shape)

Cell 9: Train / val split for CNN-token sequences

In [ ]:
# Cell 9: Train / val split for CNN-token sequences

VAL_RATIO = 0.15
BATCH_SIZE = 32
MAX_LEN = 120

train_indices, val_indices = train_test_split(
    np.arange(len(X_train_frames)),
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=y_train,
)

seqid_train_used = np.array(seqid_train_used, dtype=object)
seqid_test_used = np.array(seqid_test_used, dtype=object)

X_subtrain_frames = [X_train_frames[i] for i in train_indices]
y_subtrain = y_train[train_indices]
seqid_subtrain = seqid_train_used[train_indices]

X_val_frames = [X_train_frames[i] for i in val_indices]
y_val = y_train[val_indices]
seqid_val = seqid_train_used[val_indices]

print("Subtrain size:", len(X_subtrain_frames))
print("Val size:", len(X_val_frames))
print("Test size:", len(X_test_frames))
print("MAX_LEN:", MAX_LEN)

Cell 10: Dataset / DataLoader with dynamic padding

In [ ]:
# Cell 10: Dataset / DataLoader with dynamic padding

class TOFFrameSequenceDataset(Dataset):
    def __init__(self, X_frames, y, seqids):
        self.X_frames = X_frames
        self.y = np.array(y, dtype=np.int64)
        self.seqids = np.array(seqids, dtype=object)
        assert len(self.X_frames) == len(self.y) == len(self.seqids)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "frames": self.X_frames[idx],   # (T, 5, 8, 8)
            "label": int(self.y[idx]),
            "seqid": self.seqids[idx],
        }


def collate_tof_frame_batch(batch, max_len=MAX_LEN):
    lengths = [min(item["frames"].shape[0], max_len) for item in batch]
    batch_max_len = max(lengths)
    B = len(batch)

    padded = torch.zeros((B, batch_max_len, 5, 8, 8), dtype=torch.float32)
    labels = torch.zeros((B,), dtype=torch.long)
    seqids = []

    for i, item in enumerate(batch):
        frames = item["frames"][:max_len]
        L = frames.shape[0]
        padded[i, :L] = torch.from_numpy(frames)
        labels[i] = item["label"]
        seqids.append(item["seqid"])

    lengths = torch.tensor(lengths, dtype=torch.long)

    return {
        "input_frames": padded,
        "length": lengths,
        "label": labels,
        "seqid": np.array(seqids, dtype=object),
    }

train_dataset = TOFFrameSequenceDataset(X_subtrain_frames, y_subtrain, seqid_subtrain)
val_dataset = TOFFrameSequenceDataset(X_val_frames, y_val, seqid_val)
test_dataset = TOFFrameSequenceDataset(X_test_frames, y_test, seqid_test_used)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda batch: collate_tof_frame_batch(batch, max_len=MAX_LEN),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: collate_tof_frame_batch(batch, max_len=MAX_LEN),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: collate_tof_frame_batch(batch, max_len=MAX_LEN),
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

sample_batch = next(iter(train_loader))
print("sample input_frames shape:", sample_batch["input_frames"].shape)
print("sample lengths shape:", sample_batch["length"].shape)

Cell 11: Define CNN-token + BiLSTM model

In [ ]:
# Cell 11: Define CNN-token + BiLSTM model

class FrameCNNEncoder(nn.Module):
    def __init__(self, out_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(5, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 8 -> 4
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 4 -> 2
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.proj(x)
        return x


class CNNTokenBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        token_dim: int,
        hidden_dim: int,
        num_classes: int,
        num_layers: int = 1,
        dropout: float = 0.3,
        bidirectional: bool = True,
    ):
        super().__init__()
        self.encoder = FrameCNNEncoder(out_dim=token_dim, dropout=dropout)
        self.lstm = nn.LSTM(
            input_size=token_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_frames, lengths):
        B, L, C, H, W = input_frames.shape
        x = input_frames.view(B * L, C, H, W)
        token_emb = self.encoder(x).view(B, L, -1)  # (B, L, token_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            token_emb,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (h_n, _) = self.lstm(packed)

        if self.lstm.bidirectional:
            last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            last_hidden = h_n[-1]

        logits = self.classifier(self.dropout(last_hidden))
        return logits


TOKEN_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = len(classes)

model = CNNTokenBiLSTMClassifier(
    token_dim=TOKEN_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=1,
    dropout=0.3,
    bidirectional=True,
).to(DEVICE)

print(model)

Cell 12: Training setup

In [ ]:
# Cell 12: Training setup

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6
GRAD_CLIP_NORM = 1.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print("LEARNING_RATE:", LEARNING_RATE)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("EPOCHS:", EPOCHS)
print("EARLY_STOPPING_PATIENCE:", EARLY_STOPPING_PATIENCE)

Cell 14: Define BFRB mapping + binary threshold helpers

In [ ]:
# Cell 14: Define BFRB mapping + binary threshold helpers

BFRB_CLASS_NAMES = {
    "Above ear - pull hair",
    "Cheek - pinch skin",
    "Eyebrow - pull hair",
    "Eyelash - pull hair",
    "Forehead - pull hairline",
    "Forehead - scratch",
    "Neck - pinch skin",
    "Neck - scratch",
}

bfrb_class_ids = sorted([label2id[name] for name in BFRB_CLASS_NAMES if name in label2id])

print("BFRB class ids:", bfrb_class_ids)
print("BFRB class names:", [id2label[i] for i in bfrb_class_ids])

def to_binary_labels(y_multiclass, positive_ids):
    """
    18-class -> binary
    BFRB = 1, non-BFRB = 0
    """
    return np.array([1 if y in positive_ids else 0 for y in y_multiclass], dtype=np.int64)

def to_9class_labels(y_multiclass, bfrb_ids):
    """
    18-class -> 9-class (paper setting)
    - 8 BFRB classes keep their original ids
    - all non-BFRB classes collapse into one shared class id = 8
    """
    bfrb_ids = list(sorted(bfrb_ids))
    bfrb_id_to_9class = {orig_id: new_id for new_id, orig_id in enumerate(bfrb_ids)}
    non_bfrb_class_id = len(bfrb_ids)  # should be 8

    y_new = []
    for y in y_multiclass:
        if y in bfrb_id_to_9class:
            y_new.append(bfrb_id_to_9class[y])
        else:
            y_new.append(non_bfrb_class_id)

    return np.array(y_new, dtype=np.int64)

def binary_probs_from_logits(logits: np.ndarray, positive_ids):
    """
    logits: (N, C)
    return:
      binary_probs: (N,) = sum of softmax probs over BFRB classes
    """
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probs[:, positive_ids].sum(axis=1)

def find_best_binary_threshold(binary_targets, binary_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.951, 0.01)

    best_threshold = 0.50
    best_f1 = -1.0

    for thr in thresholds:
        preds = (binary_probs >= thr).astype(np.int64)
        f1 = f1_score(binary_targets, preds, average="binary")

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = float(thr)

    return best_threshold, best_f1

# quick sanity check
sample_y = np.array([0, 1, 2, 10, 15])
print("sample binary:", to_binary_labels(sample_y, bfrb_class_ids))
print("sample 9-class:", to_9class_labels(sample_y, bfrb_class_ids))

Cell 15: Train one epoch

In [ ]:
# Cell 15: Train one epoch

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_samples = 0

    all_preds = []
    all_targets = []

    for batch in loader:
        input_frames = batch["input_frames"].to(device)
        lengths = batch["length"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_frames, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(labels.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / total_samples
    y_true_9 = to_9class_labels(all_targets, bfrb_class_ids)
    y_pred_9 = to_9class_labels(all_preds, bfrb_class_ids)
    macro_f1_9class = f1_score(y_true_9, y_pred_9, average="macro")

    binary_targets = to_binary_labels(all_targets, bfrb_class_ids)
    binary_preds = to_binary_labels(all_preds, bfrb_class_ids)
    binary_f1 = f1_score(binary_targets, binary_preds, average="binary")

    macro_f1_18class = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, macro_f1_9class, binary_f1, macro_f1_18class

Cell 16: Evaluate model

In [ ]:
# Cell 16: Evaluate model

def evaluate(model, loader, criterion, device, binary_threshold=None, tune_binary_threshold=False):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    all_logits = []
    all_preds = []
    all_targets = []
    all_seqids = []

    with torch.no_grad():
        for batch in loader:
            input_frames = batch["input_frames"].to(device)
            lengths = batch["length"].to(device)
            labels = batch["label"].to(device)
            seqids = batch["seqid"]

            logits = model(input_frames, lengths)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)
            all_logits.append(logits.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(labels.detach().cpu().numpy())
            all_seqids.extend(list(seqids))

    all_logits = np.concatenate(all_logits, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    avg_loss = total_loss / total_samples

    y_true_9 = to_9class_labels(all_targets, bfrb_class_ids)
    y_pred_9 = to_9class_labels(all_preds, bfrb_class_ids)
    macro_f1_9class = f1_score(y_true_9, y_pred_9, average="macro")

    binary_targets = to_binary_labels(all_targets, bfrb_class_ids)
    binary_preds_argmax = to_binary_labels(all_preds, bfrb_class_ids)
    binary_f1_argmax = f1_score(binary_targets, binary_preds_argmax, average="binary")

    binary_probs = binary_probs_from_logits(all_logits, bfrb_class_ids)
    if tune_binary_threshold:
        binary_threshold, binary_f1 = find_best_binary_threshold(binary_targets, binary_probs)
    else:
        if binary_threshold is None:
            binary_threshold = 0.50
        binary_preds = (binary_probs >= binary_threshold).astype(np.int64)
        binary_f1 = f1_score(binary_targets, binary_preds, average="binary")

    macro_f1_18class = f1_score(all_targets, all_preds, average="macro")
    return {
        "loss": avg_loss,
        "macro_f1_9class": macro_f1_9class,
        "binary_f1": binary_f1,
        "binary_f1_argmax": binary_f1_argmax,
        "binary_threshold": float(binary_threshold),
        "binary_probs": binary_probs,
        "macro_f1_18class": macro_f1_18class,
        "logits": all_logits,
        "preds": all_preds,
        "targets": all_targets,
        "seqids": np.array(all_seqids, dtype=object),
    }

Cell 17: Train loop (validation threshold tuning + early stopping)

In [ ]:
# Cell 17: Train loop (validation threshold tuning + early stopping)

history = []

best_val_binary_f1 = -1.0
best_binary_threshold = 0.50
best_state_dict = None
best_val_eval = None
best_test_eval = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_macro_f1_9class, train_binary_f1, train_macro_f1_18class = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )

    val_result = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE,
        tune_binary_threshold=True,
    )

    test_result = evaluate(
        model,
        test_loader,
        criterion,
        DEVICE,
        binary_threshold=val_result["binary_threshold"],
        tune_binary_threshold=False,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1_9class": train_macro_f1_9class,
        "train_binary_f1_argmax": train_binary_f1,
        "train_macro_f1_18class": train_macro_f1_18class,
        "val_loss": val_result["loss"],
        "val_macro_f1_9class": val_result["macro_f1_9class"],
        "val_binary_f1": val_result["binary_f1"],
        "val_binary_f1_argmax": val_result["binary_f1_argmax"],
        "val_binary_threshold": val_result["binary_threshold"],
        "val_macro_f1_9class": val_result["macro_f1_18class"],
        "test_loss": test_result["loss"],
        "test_macro_f1_9class": test_result["macro_f1_9class"],
        "test_binary_f1": test_result["binary_f1"],
        "test_binary_f1_argmax": test_result["binary_f1_argmax"],
        "test_binary_threshold": test_result["binary_threshold"],
        "test_macro_f1_9class": test_result["macro_f1_18class"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_binary_f1_argmax={train_binary_f1:.4f} | "
        f"val_binary_f1={val_result['binary_f1']:.4f} @thr={val_result['binary_threshold']:.2f} | "
        f"test_binary_f1={test_result['binary_f1']:.4f} | "
        f"val_macro_f1_9class={val_result['macro_f1_9class']:.4f} | "
        f"test_macro_f1_9class={test_result['macro_f1_9class']:.4f}"
    )

    if val_result["binary_f1"] > best_val_binary_f1:
        best_val_binary_f1 = val_result["binary_f1"]
        best_binary_threshold = val_result["binary_threshold"]
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_val_eval = val_result
        best_test_eval = test_result
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

Cell 18: Load best model and finalize evaluation

In [ ]:
# Cell 18: Load best model and finalize evaluation

assert best_state_dict is not None, "No best model was saved."

model.load_state_dict(best_state_dict)

final_val_eval = evaluate(
    model,
    val_loader,
    criterion,
    DEVICE,
    tune_binary_threshold=True,
)

final_eval = evaluate(
    model,
    test_loader,
    criterion,
    DEVICE,
    binary_threshold=best_binary_threshold,
    tune_binary_threshold=False,
)

print("\nFinal validation results (best checkpoint):")
print("val_loss:", final_val_eval["loss"])
print("val_macro_f1_9class:", final_val_eval["macro_f1_9class"])
print("val_binary_f1:", final_val_eval["binary_f1"])
print("val_binary_f1_argmax:", final_val_eval["binary_f1_argmax"])
#print("val_macro_f1_9class:", final_val_eval["macro_f1_18class"])
print("best_binary_threshold:", best_binary_threshold)

print("\nFinal test results (best model):")
print("test_loss:", final_eval["loss"])
print("test_macro_f1_9class:", final_eval["macro_f1_9class"])
print("test_binary_f1:", final_eval["binary_f1"])
print("test_binary_f1_argmax:", final_eval["binary_f1_argmax"])
#print("test_macro_f1_9class:", final_eval["macro_f1_18class"])
print("binary_threshold_used:", final_eval["binary_threshold"])
print("logits shape:", final_eval["logits"].shape)
print("targets shape:", final_eval["targets"].shape)
print("seqids shape:", final_eval["seqids"].shape)

Cell 19: output some files

In [ ]:
# Cell 19: output some files

import os
import json
import numpy as np
from sklearn.metrics import accuracy_score

acc = accuracy_score(final_eval["targets"], final_eval["preds"])
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
MODEL_DIR = os.path.join(PROJECT_ROOT, "results", "Model12_Training")
os.makedirs(MODEL_DIR, exist_ok=True)

OUT_LOGITS = os.path.join(MODEL_DIR, "Model_12_TOF_Only_CNN_Token_LSTM_test_logits.npy")
OUT_Y = os.path.join(MODEL_DIR, "Model_12_TOF_Only_CNN_Token_LSTM_test_y.npy")
OUT_SEQ_IDS = os.path.join(MODEL_DIR, "Model_12_TOF_Only_CNN_Token_LSTM_test_seqids.npy")
OUT_METRICS = os.path.join(MODEL_DIR, "model12_training_summary.json")

np.save(OUT_LOGITS, final_eval["logits"])
np.save(OUT_Y, final_eval["targets"])
np.save(OUT_SEQ_IDS, final_eval["seqids"])

metrics_to_save = {
    #"epochs": int(len(history)),
    #"best_binary_threshold": float(best_binary_threshold),
    "final_metrics": {
        "acc": float(acc),
        "macroF1": float(final_eval["macro_f1_9class"]),
        "binaryF1": float(final_eval["binary_f1"]),
    }
}

with open(OUT_METRICS, "w") as f:
    json.dump(metrics_to_save, f, indent=4)

print("Saved to:", MODEL_DIR)

Cell 20: Training history

In [ ]:
# Cell 20: Training history

history_df = pd.DataFrame(history)
display(history_df)

Cell 21: save training curves for Model 12

In [ ]:
# Cell 21: save training curves for Model 12

import os
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("default")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PLOT_DIR = os.path.join(PROJECT_ROOT, "plots", "model_12_training_curves")
os.makedirs(PLOT_DIR, exist_ok=True)

history_df = pd.DataFrame(history)
print("history_df columns:", history_df.columns.tolist())
print("Saving plots to:", PLOT_DIR)

plt.figure(figsize=(8, 5))
if "train_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
if "val_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
if "test_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_loss"], label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model 12 Loss Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "loss_curve.png"), dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
if "train_binary_f1_argmax" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_binary_f1_argmax"], label="Train Binary F1 (Argmax)")
if "val_binary_f1" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_binary_f1"], label="Val Binary F1")
if "test_binary_f1" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_binary_f1"], label="Test Binary F1")
plt.xlabel("Epoch")
plt.ylabel("Binary F1")
plt.title("Model 12 Binary F1 Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "binary_f1_curve.png"), dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
if "train_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_macro_f1_9class"], label="Train Macro F1 (9-class)")
if "val_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_macro_f1_9class"], label="Val Macro F1 (9-class)")
if "test_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_macro_f1_9class"], label="Test Macro F1 (9-class)")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Model 12 Macro F1 Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "macro_f1_curve.png"), dpi=200)
plt.close()

print("Saved:", os.path.join(PLOT_DIR, "loss_curve.png"))
print("Saved:", os.path.join(PLOT_DIR, "binary_f1_curve.png"))
print("Saved:", os.path.join(PLOT_DIR, "macro_f1_curve.png"))